# Classical Parameter Tuning

This notebook tunes the **classical image-processing parameters** for Mode A. It is parameter tuning, not machine-learning training.

The 30 annotated images are split reproducibly into:

- **21 tuning images (70%)** — used to compare candidate parameter values.
- **9 final-evaluation images (30%)** — kept untouched until the second notebook.

The notebook **does not edit `config.py` automatically**. It only ranks candidate settings and prints a recommended set. After reviewing the result, copy the selected values into `banana_quality/config.py` yourself.

Recommended workflow:

1. Tune banana `K` + cluster acceptance.
2. Tune banana opening + closing.
3. Freeze the banana settings.
4. Tune blemish `K` + colour-distance threshold.
5. Tune darkness threshold.
6. Tune blemish opening + closing.
7. Copy the final recommended values into `config.py`.

The remaining internal thresholds are kept fixed to reduce overfitting on only 30 images.

In [1]:
from pathlib import Path
from dataclasses import replace
from itertools import product
import random
import sys

import cv2
import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the project root whether Jupyter starts here or in a subfolder."""
    for candidate in [start, *start.parents]:
        if (candidate / "banana_quality" / "config.py").exists() and (candidate / "data" / "annotations").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root. Start Jupyter from inside this project.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from banana_quality.config import ProcessingConfig
from banana_quality.segmentation.kmeans import prepare_image
from banana_quality.segmentation.mask_refinement import banana_mask_from_prepared
from banana_quality.blemish.detection import blemish_mask_from_prepared
from banana_quality.surface_quality import blemish_percentage

DATASET_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_DIR = PROJECT_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)

Project root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system
Dataset root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\data\annotations


## 1. Create a fixed 21/9 random split

The 30 annotated images are treated as one pool. A fixed random seed is used to select **21 tuning images** and **9 final-evaluation images**. No `banana`, `freshunripe`, or `mold` filename grouping is used.

Because the same seed and procedure are used in the final-evaluation notebook, the split is reproducible.

In [2]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT_SEED = 42
FINAL_EVALUATION_COUNT = 9

images_dir = DATASET_ROOT / "images"
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

# Treat all annotated images as one pool and shuffle reproducibly.
shuffled_paths = image_paths.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(shuffled_paths)

# Keep exactly 9 images untouched for final evaluation.
final_names = {p.name for p in shuffled_paths[:FINAL_EVALUATION_COUNT]}

split_rows = [
    {
        "image": path.name,
        "split": "final_evaluation" if path.name in final_names else "tuning",
    }
    for path in image_paths
]

split_df = pd.DataFrame(split_rows).sort_values(["split", "image"]).reset_index(drop=True)
split_df.to_csv(RESULTS_DIR / "dataset_split.csv", index=False)

print(split_df["split"].value_counts())
print("\nSaved split:", RESULTS_DIR / "dataset_split.csv")

assert len(image_paths) == 30, f"Expected 30 images, found {len(image_paths)}"
assert int((split_df["split"] == "tuning").sum()) == 21
assert int((split_df["split"] == "final_evaluation").sum()) == 9

split
tuning              21
final_evaluation     9
Name: count, dtype: int64

Saved split: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\evaluation\results\dataset_split.csv


## 2. Load tuning images and metrics

In [3]:
def matching_mask(folder: Path, image_path: Path) -> Path:
    path = folder / f"{image_path.stem}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_binary_mask(path: Path, target_shape: tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {path}")
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)
    if mask.shape != target_shape:
        h, w = target_shape
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


def resize_binary_mask(mask: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    h, w = target_shape
    resized = cv2.resize(np.asarray(mask, dtype=np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    return np.where(resized > 0, 255, 0).astype(np.uint8)


def iou_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    union = int(np.logical_or(pred, gt).sum())
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / union)


def dice_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    denominator = int(pred.sum() + gt.sum())
    if denominator == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denominator)


def load_samples(split_name: str):
    names = set(split_df.loc[split_df["split"] == split_name, "image"])
    samples = []
    for image_path in image_paths:
        if image_path.name not in names:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Cannot read image: {image_path}")
        shape = image.shape[:2]
        banana_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "banana_masks", image_path), shape
        )
        blemish_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "blemish_masks", image_path), shape
        )
        samples.append({
            "name": image_path.name,
            "image": image,
            "shape": shape,
            "banana_gt": banana_gt,
            "blemish_gt": blemish_gt,
        })
    return samples


tuning_samples = load_samples("tuning")
print("Loaded tuning samples:", len(tuning_samples))

Loaded tuning samples: 21


## 3. Tune banana K + cluster acceptance

Only these two parameters change in this stage. Banana morphology stays at the current default while the best K/acceptance pair is identified.

You can edit the candidate lists if you want a narrower or wider search.

In [4]:
DEFAULTS = ProcessingConfig()

BANANA_K_VALUES = [4, 5, 6, 7, 8]
BANANA_ACCEPTANCE_VALUES = [0.35, 0.40, 0.45, 0.50, 0.55]

BASE_BANANA_OPENING = int(DEFAULTS.banana_opening_kernel)
BASE_BANANA_CLOSING = int(DEFAULTS.banana_closing_kernel)


def evaluate_banana_configuration(k: int, acceptance: float, opening: int, closing: int) -> dict:
    config = replace(
        DEFAULTS,
        kmeans_k=int(k),
        banana_cluster_score_threshold=float(acceptance),
    )
    ious, dices = [], []

    for sample in tuning_samples:
        prepared = prepare_image(sample["image"], config)
        banana_steps = banana_mask_from_prepared(
            prepared,
            opening_kernel_size=int(opening),
            closing_kernel_size=int(closing),
        )
        pred = resize_binary_mask(banana_steps["banana_mask"], sample["shape"])
        ious.append(iou_score(pred, sample["banana_gt"]))
        dices.append(dice_score(pred, sample["banana_gt"]))

    return {
        "kmeans_k": int(k),
        "banana_cluster_score_threshold": float(acceptance),
        "banana_opening_kernel": int(opening),
        "banana_closing_kernel": int(closing),
        "banana_mean_iou": float(np.mean(ious)),
        "banana_mean_dice": float(np.mean(dices)),
    }


banana_pair_rows = []
combinations = list(product(BANANA_K_VALUES, BANANA_ACCEPTANCE_VALUES))
for index, (k, acceptance) in enumerate(combinations, start=1):
    print(f"Banana K/acceptance {index}/{len(combinations)}: K={k}, acceptance={acceptance:.2f}")
    banana_pair_rows.append(
        evaluate_banana_configuration(
            k, acceptance, BASE_BANANA_OPENING, BASE_BANANA_CLOSING
        )
    )

banana_pair_results = pd.DataFrame(banana_pair_rows).sort_values(
    ["banana_mean_dice", "banana_mean_iou"], ascending=[False, False]
).reset_index(drop=True)

banana_pair_results.to_csv(RESULTS_DIR / "tuning_banana_k_acceptance.csv", index=False)
banana_pair_results.head(10)

Banana K/acceptance 1/25: K=4, acceptance=0.35
Banana K/acceptance 2/25: K=4, acceptance=0.40
Banana K/acceptance 3/25: K=4, acceptance=0.45
Banana K/acceptance 4/25: K=4, acceptance=0.50
Banana K/acceptance 5/25: K=4, acceptance=0.55
Banana K/acceptance 6/25: K=5, acceptance=0.35
Banana K/acceptance 7/25: K=5, acceptance=0.40
Banana K/acceptance 8/25: K=5, acceptance=0.45
Banana K/acceptance 9/25: K=5, acceptance=0.50
Banana K/acceptance 10/25: K=5, acceptance=0.55
Banana K/acceptance 11/25: K=6, acceptance=0.35
Banana K/acceptance 12/25: K=6, acceptance=0.40
Banana K/acceptance 13/25: K=6, acceptance=0.45
Banana K/acceptance 14/25: K=6, acceptance=0.50
Banana K/acceptance 15/25: K=6, acceptance=0.55
Banana K/acceptance 16/25: K=7, acceptance=0.35
Banana K/acceptance 17/25: K=7, acceptance=0.40
Banana K/acceptance 18/25: K=7, acceptance=0.45
Banana K/acceptance 19/25: K=7, acceptance=0.50
Banana K/acceptance 20/25: K=7, acceptance=0.55
Banana K/acceptance 21/25: K=8, acceptance=0.35
B

,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,banana_mean_iou,banana_mean_dice
0,7,0.35,3,5,0.925407,0.959480
1,6,0.35,3,5,0.920241,0.956109
2,6,0.40,3,5,0.917320,0.954534
3,7,0.40,3,5,0.912120,0.951309
4,7,0.45,3,5,0.912120,0.951309
5,5,0.35,3,5,0.901245,0.945874
6,8,0.35,3,5,0.904950,0.945508
7,8,0.40,3,5,0.904950,0.945508
8,8,0.45,3,5,0.904950,0.945508
9,7,0.55,3,5,0.900979,0.943731


## 4. Tune banana morphology

The best K/acceptance pair from the previous table is frozen. This stage compares opening and closing kernels.

In [5]:
best_banana_pair = banana_pair_results.iloc[0]
BEST_BANANA_K = int(best_banana_pair["kmeans_k"])
BEST_BANANA_ACCEPTANCE = float(best_banana_pair["banana_cluster_score_threshold"])

BANANA_OPENING_VALUES = [1, 3, 5]
BANANA_CLOSING_VALUES = [1, 3, 5, 7]

banana_morph_rows = []
for opening, closing in product(BANANA_OPENING_VALUES, BANANA_CLOSING_VALUES):
    banana_morph_rows.append(
        evaluate_banana_configuration(
            BEST_BANANA_K, BEST_BANANA_ACCEPTANCE, opening, closing
        )
    )

banana_morph_results = pd.DataFrame(banana_morph_rows).sort_values(
    ["banana_mean_dice", "banana_mean_iou"], ascending=[False, False]
).reset_index(drop=True)

banana_morph_results.to_csv(RESULTS_DIR / "tuning_banana_morphology.csv", index=False)
banana_morph_results.head(10)

,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,banana_mean_iou,banana_mean_dice
0,7,0.35,3,7,0.927202,0.960584
1,7,0.35,3,5,0.925407,0.959480
2,7,0.35,5,7,0.925397,0.959420
3,7,0.35,3,3,0.923342,0.958333
4,7,0.35,5,5,0.923249,0.958136
5,7,0.35,3,1,0.922496,0.957770
6,7,0.35,5,3,0.919247,0.955732
7,7,0.35,5,1,0.917383,0.954729
8,7,0.35,1,7,0.919428,0.954637
9,7,0.35,1,3,0.917879,0.954210


## 5. Freeze the best banana settings

Blemish tuning should use one fixed banana segmentation. This avoids mixing banana errors with constantly changing blemish settings.

In [6]:
best_banana = banana_morph_results.iloc[0]
BEST_BANANA_K = int(best_banana["kmeans_k"])
BEST_BANANA_ACCEPTANCE = float(best_banana["banana_cluster_score_threshold"])
BEST_BANANA_OPENING = int(best_banana["banana_opening_kernel"])
BEST_BANANA_CLOSING = int(best_banana["banana_closing_kernel"])

print("Best banana settings on tuning set:")
print({
    "kmeans_k": BEST_BANANA_K,
    "banana_cluster_score_threshold": BEST_BANANA_ACCEPTANCE,
    "banana_opening_kernel": BEST_BANANA_OPENING,
    "banana_closing_kernel": BEST_BANANA_CLOSING,
    "banana_mean_iou": float(best_banana["banana_mean_iou"]),
    "banana_mean_dice": float(best_banana["banana_mean_dice"]),
})

# Cache the fixed banana result once per image so blemish tuning is faster.
best_banana_config = replace(
    DEFAULTS,
    kmeans_k=BEST_BANANA_K,
    banana_cluster_score_threshold=BEST_BANANA_ACCEPTANCE,
)

blemish_cache = []
for sample in tuning_samples:
    prepared = prepare_image(sample["image"], best_banana_config)
    banana_steps = banana_mask_from_prepared(
        prepared,
        opening_kernel_size=BEST_BANANA_OPENING,
        closing_kernel_size=BEST_BANANA_CLOSING,
    )
    banana_working = np.asarray(banana_steps["banana_mask"], dtype=np.uint8)
    banana_original = resize_binary_mask(banana_working, sample["shape"])
    blemish_cache.append({
        **sample,
        "prepared": prepared,
        "banana_working": banana_working,
        "banana_pred": banana_original,
    })

print("Cached fixed banana results for", len(blemish_cache), "tuning images.")

Best banana settings on tuning set:
{'kmeans_k': 7, 'banana_cluster_score_threshold': 0.35, 'banana_opening_kernel': 3, 'banana_closing_kernel': 7, 'banana_mean_iou': 0.9272024356343475, 'banana_mean_dice': 0.9605840001931989}
Cached fixed banana results for 21 tuning images.


## 6. Tune blemish K + colour sensitivity

The second K-means uses `blemish_kmeans_k`. The colour sensitivity here is the Lab a,b chroma-distance threshold (`blemish_cluster_chroma_threshold`).

Ranking priority is:

1. Higher blemish Dice
2. Higher blemish IoU
3. Lower blemish-percentage MAE

In [7]:
BASE_BLEMISH_OPENING = int(DEFAULTS.blemish_opening_kernel)
BASE_BLEMISH_CLOSING = int(DEFAULTS.blemish_closing_kernel)


def evaluate_blemish_configuration(
    blemish_k: int,
    chroma_threshold: float,
    darkness_threshold: float,
    opening: int,
    closing: int,
) -> dict:
    config = replace(
        best_banana_config,
        blemish_kmeans_k=int(blemish_k),
        blemish_cluster_chroma_threshold=float(chroma_threshold),
        darkness_threshold=float(darkness_threshold),
    )

    ious, dices, percentage_errors = [], [], []

    for sample in blemish_cache:
        steps = blemish_mask_from_prepared(
            sample["prepared"],
            sample["banana_working"],
            opening_kernel_size=int(opening),
            closing_kernel_size=int(closing),
            config=config,
        )
        pred = resize_binary_mask(steps["blemish_mask"], sample["shape"])

        ious.append(iou_score(pred, sample["blemish_gt"]))
        dices.append(dice_score(pred, sample["blemish_gt"]))

        predicted_pct = blemish_percentage(sample["banana_pred"], pred)
        ground_truth_pct = blemish_percentage(sample["banana_gt"], sample["blemish_gt"])
        percentage_errors.append(abs(predicted_pct - ground_truth_pct))

    return {
        "blemish_kmeans_k": int(blemish_k),
        "blemish_cluster_chroma_threshold": float(chroma_threshold),
        "darkness_threshold": float(darkness_threshold),
        "blemish_opening_kernel": int(opening),
        "blemish_closing_kernel": int(closing),
        "blemish_mean_iou": float(np.mean(ious)),
        "blemish_mean_dice": float(np.mean(dices)),
        "blemish_percentage_mae": float(np.mean(percentage_errors)),
    }


def rank_blemish(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(
        ["blemish_mean_dice", "blemish_mean_iou", "blemish_percentage_mae"],
        ascending=[False, False, True],
    ).reset_index(drop=True)


BLEMISH_K_VALUES = [3, 4, 5, 6, 7]
BLEMISH_CHROMA_VALUES = [12.0, 15.0, 18.0, 21.0, 24.0]

blemish_pair_rows = []
combinations = list(product(BLEMISH_K_VALUES, BLEMISH_CHROMA_VALUES))
for index, (k, chroma) in enumerate(combinations, start=1):
    print(f"Blemish K/chroma {index}/{len(combinations)}: K={k}, chroma={chroma:.1f}")
    blemish_pair_rows.append(
        evaluate_blemish_configuration(
            k, chroma, DEFAULTS.darkness_threshold,
            BASE_BLEMISH_OPENING, BASE_BLEMISH_CLOSING
        )
    )

blemish_pair_results = rank_blemish(pd.DataFrame(blemish_pair_rows))
blemish_pair_results.to_csv(RESULTS_DIR / "tuning_blemish_k_chroma.csv", index=False)
blemish_pair_results.head(10)

Blemish K/chroma 1/25: K=3, chroma=12.0
Blemish K/chroma 2/25: K=3, chroma=15.0
Blemish K/chroma 3/25: K=3, chroma=18.0
Blemish K/chroma 4/25: K=3, chroma=21.0
Blemish K/chroma 5/25: K=3, chroma=24.0
Blemish K/chroma 6/25: K=4, chroma=12.0
Blemish K/chroma 7/25: K=4, chroma=15.0
Blemish K/chroma 8/25: K=4, chroma=18.0
Blemish K/chroma 9/25: K=4, chroma=21.0
Blemish K/chroma 10/25: K=4, chroma=24.0
Blemish K/chroma 11/25: K=5, chroma=12.0
Blemish K/chroma 12/25: K=5, chroma=15.0
Blemish K/chroma 13/25: K=5, chroma=18.0
Blemish K/chroma 14/25: K=5, chroma=21.0
Blemish K/chroma 15/25: K=5, chroma=24.0
Blemish K/chroma 16/25: K=6, chroma=12.0
Blemish K/chroma 17/25: K=6, chroma=15.0
Blemish K/chroma 18/25: K=6, chroma=18.0
Blemish K/chroma 19/25: K=6, chroma=21.0
Blemish K/chroma 20/25: K=6, chroma=24.0
Blemish K/chroma 21/25: K=7, chroma=12.0
Blemish K/chroma 22/25: K=7, chroma=15.0
Blemish K/chroma 23/25: K=7, chroma=18.0
Blemish K/chroma 24/25: K=7, chroma=21.0
Blemish K/chroma 25/25: K

,blemish_kmeans_k,blemish_cluster_chroma_threshold,darkness_threshold,blemish_opening_kernel,blemish_closing_kernel,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,6,21.0,24.0,3,5,0.383152,0.494050,7.641335
1,7,24.0,24.0,3,5,0.376588,0.482593,5.825102
2,6,24.0,24.0,3,5,0.372083,0.478471,7.171286
3,3,24.0,24.0,3,5,0.370373,0.471285,8.029402
4,3,21.0,24.0,3,5,0.366110,0.464182,8.172955
5,5,24.0,24.0,3,5,0.356653,0.463441,9.209322
6,4,24.0,24.0,3,5,0.353041,0.459339,8.839456
7,4,21.0,24.0,3,5,0.341884,0.447087,9.986453
8,7,15.0,24.0,3,5,0.329541,0.444110,8.995879
9,7,18.0,24.0,3,5,0.334987,0.442659,6.008685


## 7. Tune darkness threshold

The best blemish K/chroma pair is frozen. This stage changes only `darkness_threshold`, which belongs to supplemental Rule A.

In [8]:
best_blemish_pair = blemish_pair_results.iloc[0]
BEST_BLEMISH_K = int(best_blemish_pair["blemish_kmeans_k"])
BEST_BLEMISH_CHROMA = float(best_blemish_pair["blemish_cluster_chroma_threshold"])

DARKNESS_VALUES = [16.0, 20.0, 24.0, 28.0, 32.0]

darkness_rows = [
    evaluate_blemish_configuration(
        BEST_BLEMISH_K, BEST_BLEMISH_CHROMA, darkness,
        BASE_BLEMISH_OPENING, BASE_BLEMISH_CLOSING
    )
    for darkness in DARKNESS_VALUES
]

darkness_results = rank_blemish(pd.DataFrame(darkness_rows))
darkness_results.to_csv(RESULTS_DIR / "tuning_blemish_darkness.csv", index=False)
darkness_results

,blemish_kmeans_k,blemish_cluster_chroma_threshold,darkness_threshold,blemish_opening_kernel,blemish_closing_kernel,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,6,21.0,32.0,3,5,0.384559,0.495121,7.657452
1,6,21.0,28.0,3,5,0.383721,0.494496,7.653496
2,6,21.0,24.0,3,5,0.383152,0.494050,7.641335
3,6,21.0,20.0,3,5,0.382463,0.493492,7.639120
4,6,21.0,16.0,3,5,0.381285,0.492457,7.653125


## 8. Tune blemish morphology

The best K, chroma threshold, and darkness threshold are now fixed. Only blemish opening and closing are compared.

In [9]:
best_darkness = darkness_results.iloc[0]
BEST_DARKNESS = float(best_darkness["darkness_threshold"])

BLEMISH_OPENING_VALUES = [1, 3, 5]
BLEMISH_CLOSING_VALUES = [1, 3, 5, 7]

blemish_morph_rows = []
for opening, closing in product(BLEMISH_OPENING_VALUES, BLEMISH_CLOSING_VALUES):
    blemish_morph_rows.append(
        evaluate_blemish_configuration(
            BEST_BLEMISH_K, BEST_BLEMISH_CHROMA, BEST_DARKNESS,
            opening, closing
        )
    )

blemish_morph_results = rank_blemish(pd.DataFrame(blemish_morph_rows))
blemish_morph_results.to_csv(RESULTS_DIR / "tuning_blemish_morphology.csv", index=False)
blemish_morph_results.head(10)

,blemish_kmeans_k,blemish_cluster_chroma_threshold,darkness_threshold,blemish_opening_kernel,blemish_closing_kernel,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,6,21.0,32.0,3,3,0.385197,0.495743,7.614507
1,6,21.0,32.0,3,1,0.385067,0.495618,7.612686
2,6,21.0,32.0,3,5,0.384559,0.495121,7.657452
3,6,21.0,32.0,3,7,0.384016,0.494658,7.735111
4,6,21.0,32.0,5,3,0.385307,0.494353,7.606055
5,6,21.0,32.0,1,3,0.382865,0.494290,7.641108
6,6,21.0,32.0,5,5,0.385173,0.494229,7.628243
7,6,21.0,32.0,5,1,0.385044,0.494081,7.604749
8,6,21.0,32.0,1,1,0.382673,0.494021,7.627804
9,6,21.0,32.0,5,7,0.384708,0.493867,7.677799


## 9. Recommended parameter set

This cell only **reports** the best tuning-set combination. It does not modify any project source file.

After you decide to use these values:

1. Open `banana_quality/config.py`.
2. Replace the corresponding defaults.
3. Restart the Jupyter kernel / Streamlit app so Python reloads the configuration.
4. Run `02_final_evaluation.ipynb` on the untouched 9-image final set.

Do not keep changing the parameters after looking at the final-evaluation result, otherwise the 9 images are no longer an unseen final set.

In [10]:
best_blemish = blemish_morph_results.iloc[0]

recommended = {
    "kmeans_k": BEST_BANANA_K,
    "banana_cluster_score_threshold": BEST_BANANA_ACCEPTANCE,
    "banana_opening_kernel": BEST_BANANA_OPENING,
    "banana_closing_kernel": BEST_BANANA_CLOSING,
    "blemish_kmeans_k": int(best_blemish["blemish_kmeans_k"]),
    "blemish_cluster_chroma_threshold": float(best_blemish["blemish_cluster_chroma_threshold"]),
    "darkness_threshold": float(best_blemish["darkness_threshold"]),
    "blemish_opening_kernel": int(best_blemish["blemish_opening_kernel"]),
    "blemish_closing_kernel": int(best_blemish["blemish_closing_kernel"]),
}

recommended_df = pd.DataFrame([recommended])
recommended_df.to_csv(RESULTS_DIR / "recommended_parameters_from_tuning.csv", index=False)
display(recommended_df)

print("\nCopy these defaults into ProcessingConfig in banana_quality/config.py:\n")
for key, value in recommended.items():
    type_name = "int" if isinstance(value, (int, np.integer)) else "float"
    print(f"    {key}: {type_name} = {value}")

print("\nSaved:", RESULTS_DIR / "recommended_parameters_from_tuning.csv")

,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,blemish_kmeans_k,blemish_cluster_chroma_threshold,darkness_threshold,blemish_opening_kernel,blemish_closing_kernel
0,7,0.35,3,7,6,21.0,32.0,3,3



Copy these defaults into ProcessingConfig in banana_quality/config.py:

    kmeans_k: int = 7
    banana_cluster_score_threshold: float = 0.35
    banana_opening_kernel: int = 3
    banana_closing_kernel: int = 7
    blemish_kmeans_k: int = 6
    blemish_cluster_chroma_threshold: float = 21.0
    darkness_threshold: float = 32.0
    blemish_opening_kernel: int = 3
    blemish_closing_kernel: int = 3

Saved: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\evaluation\results\recommended_parameters_from_tuning.csv


### Why the other config values are not tuned here

Values such as the banana HSV/Lab colour prior, healthy-cluster tolerance, Rule A/B hue/redness/yellow-loss thresholds, highlight thresholds, and tiny-component ratio remain fixed. With only 30 annotated images, tuning every threshold together would create a very large search space and increase the risk of overfitting.

If the primary tuned configuration is still clearly weak, you can later add a second controlled experiment for those internal thresholds, but they should not all be searched at once.